In [ ]:
!pip install -U transformers datasets accelerate evaluate rouge_score

In [ ]:
import transformers
import datasets
import evaluate
import accelerate
import sentencepiece
import rouge_score

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("accelerate:", accelerate.__version__)
print("sentencepiece:", sentencepiece.__version__)

transformers: 5.3.0
datasets: 4.7.0
evaluate: 0.4.6
accelerate: 1.13.0
sentencepiece: 0.2.1


In [ ]:
# TOKENIZER
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_ckpt = "facebook/bart-large"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
# =========================================================
# PREPROCESS
# =========================================================

from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")

MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):

    model_inputs = tokenizer(
    examples["article"],
    max_length=MAX_SOURCE_LENGTH,
    truncation=True,
    )

    labels = tokenizer(
    text_target=examples["highlights"],
    max_length=MAX_TARGET_LENGTH,
    truncation=True,
    )

    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing CNN/DailyMail"
)

print(tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 11490
    })
})


In [ ]:
import numpy as np
import evaluate
from transformers import DataCollatorForSeq2Seq

rouge = evaluate.load("rouge")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(pred.split(". ")) for pred in preds]
    labels = ["\n".join(label.split(". ")) for label in labels]

    return preds, labels


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    decoded_preds, decoded_labels = postprocess_text(
        decoded_preds,
        decoded_labels
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    result = {k: round(v * 100, 4) for k, v in result.items()}

    return result

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/bart_large-cnn-full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # training
    do_train=True,
    do_eval=True,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    # batch - A100 için daha verimli
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=8,   # effective batch = 256

    # logging / eval / save
    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    # best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeLsum",
    greater_is_better=True,

    # generation for ROUGE
    predict_with_generate=True,
    generation_max_length=142,
    generation_num_beams=4,

    # seq2seq stabilizers
    label_smoothing_factor=0.1,

    # precision - A100 için en doğru tercih
    bf16=True,
    fp16=False,

    # throughput
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
train_result = trainer.train()

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1000,136.291797,13.408649,2.364500,0.125100,1.967900,2.239000
2000,66.570610,8.197525,0.000000,0.000000,0.000000,0.000000
3000,65.256982,8.068940,0.000000,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/bart_large-cnn-full/tokenizer_config.json',
 '/content/drive/MyDrive/bart_large-cnn-full/tokenizer.json')

In [ ]:
from transformers import Seq2SeqTrainer

test_trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Evaluating on test set...\n")

test_metrics = test_trainer.evaluate(metric_key_prefix="test")

print("TEST ROUGE SCORES:")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

print("\n===== SAMPLE SUMMARIES =====\n")

samples = dataset["test"].select(range(3))

for i, sample in enumerate(samples):
    inputs = tokenizer(
        sample["article"],
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(model.device)

    summary_ids = model.generate(
        **inputs,
        max_length=142,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    print(f"\n====== SAMPLE {i+1} ======\n")
    print("ARTICLE:\n")
    print(sample["article"][:1000], "...\n")
    print("REFERENCE SUMMARY:\n")
    print(sample["highlights"], "\n")
    print("MODEL SUMMARY:\n")
    print(generated_summary)
    print("\n---------------------------------------")

Evaluating on test set...



RuntimeError: on_train_begin must be called before on_evaluate